# 01 · Limpieza y análisis exploratorio

**Módulo 2 · Sesión 4** — Recolección, limpieza y EDA

## Objetivos

Este es el trabajo real de un proyecto de Machine Learning: entender qué hay en los datos
antes de modelar nada. Sobre el dataset del Titanic vamos a:

1. Hacer un diagnóstico inicial: forma, tipos, diccionario de datos.
2. Detectar **columnas redundantes y fugas de datos** que vienen en el propio dataset.
3. Analizar los **valores faltantes**: cuántos, dónde y —lo importante— *por qué* faltan.
4. Revisar duplicados y outliers **sin borrarlos automáticamente**.
5. Explorar las relaciones con la variable objetivo.
6. Terminar con una lista de decisiones de preprocesamiento justificadas.

> **La regla de oro de este notebook:** no vamos a "limpiar" nada de forma automática. Cada
> decisión sobre los datos se toma con una justificación, y algunas conclusiones serán
> *dejar las cosas como están*.

## Paquetes

`pandas`, `numpy`, `matplotlib`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-2-datos-caracteristicas/notebooks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 42
pd.set_option("display.width", 100)

## 1. Primer contacto

El dataset del Titanic recoge 891 pasajeros con su información de viaje y si sobrevivieron
al naufragio. Es un clásico docente y tiene una virtud enorme para esta sesión: **está
sucio de todas las formas interesantes**.

In [ ]:
datos = pd.read_csv("../datos/titanic.csv")
print(f"Filas: {datos.shape[0]}  ·  Columnas: {datos.shape[1]}")
datos.head()

In [ ]:
datos.info()

### Diccionario de datos

Antes de tocar nada, hay que saber qué significa cada columna. Este paso se salta muy a
menudo y es la causa de la mitad de los errores posteriores.

| Columna | Significado | Tipo |
|---|---|---|
| `survived` | 1 si sobrevivió, 0 si no | **Objetivo** |
| `pclass` | Clase del billete (1, 2, 3) | Ordinal |
| `sex` | Sexo del pasajero | Categórica |
| `age` | Edad en años | Numérica |
| `sibsp` | Nº de hermanos/cónyuges a bordo | Numérica (conteo) |
| `parch` | Nº de padres/hijos a bordo | Numérica (conteo) |
| `fare` | Tarifa pagada | Numérica |
| `embarked` | Puerto de embarque (C, Q, S) | Categórica |
| `class` | Clase del billete, en texto | Categórica |
| `who` | man / woman / child | Categórica |
| `adult_male` | Si es hombre adulto | Booleana |
| `deck` | Cubierta del camarote | Categórica |
| `embark_town` | Puerto de embarque, nombre completo | Categórica |
| `alive` | yes / no | Categórica |
| `alone` | Si viajaba solo | Booleana |

Leyendo con atención esta tabla ya debería saltar una alarma. Vamos a comprobarla.

## 2. Redundancia y fugas que vienen en el dataset

Varias columnas parecen decir lo mismo que otras. Verifiquémoslo en lugar de suponerlo.

In [ ]:
print("survived vs. alive")
print(pd.crosstab(datos["survived"], datos["alive"]))

**`alive` es la variable objetivo escrita en texto.** La correspondencia es perfecta: 549
ceros son "no" y 342 unos son "yes", sin una sola excepción.

Si la dejáramos entre las variables predictoras, el modelo alcanzaría el 100 % de acierto y
sería completamente inútil: en un caso real, cuando queremos predecir si alguien
sobrevivirá, no sabemos si sobrevivió. Esto es **fuga de datos** (*data leakage*) en su
forma más pura, y viene de fábrica en el dataset.

> Es una lección importante: las fugas no siempre las introduce quien modela. A veces ya
> están en la tabla que te entregan, con otro nombre.

In [ ]:
print("pclass vs. class")
print(pd.crosstab(datos["pclass"], datos["class"]))
print("\nembarked vs. embark_town")
print(pd.crosstab(datos["embarked"], datos["embark_town"]))

`class` y `embark_town` son **duplicados exactos** de `pclass` y `embarked`, solo que en
texto. No son fuga —no contienen la respuesta— pero sí redundancia: aportan cero información
nueva y sí complejidad al preprocesamiento.

Queda `who` y `adult_male`, que parecen derivarse de `sex` y `age`.

In [ ]:
print(pd.crosstab(datos["who"], datos["adult_male"]))
print("\nEdad por categoría de 'who':")
print(datos.groupby("who")["age"].agg(["min", "max", "count"]).round(2).to_string())

Confirmado: `who` se construye a partir de `sex` y `age` (los menores de 16 son `child`), y
`adult_male` es simplemente `who == "man"`. Son **características derivadas** que alguien
ya construyó.

No son inútiles —de hecho `who` resume bien la regla "mujeres y niños primero"— pero hay que
saber que no son información independiente.

### Decisión

| Columna | Decisión | Motivo |
|---|---|---|
| `alive` | **Eliminar** | Fuga de datos: es la variable objetivo |
| `class` | Eliminar | Duplicado exacto de `pclass` |
| `embark_town` | Eliminar | Duplicado exacto de `embarked` |
| `adult_male` | Eliminar | Derivada de `who` |
| `who`, `alone` | Conservar | Derivadas, pero informativas y ya construidas |

In [ ]:
columnas_a_eliminar = ["alive", "class", "embark_town", "adult_male"]
datos_limpios = datos.drop(columns=columnas_a_eliminar)
print(f"De {datos.shape[1]} a {datos_limpios.shape[1]} columnas")
print(f"Conservadas: {list(datos_limpios.columns)}")

## 3. Valores faltantes

La pregunta no es solo *cuántos* faltan, sino **por qué** faltan. De eso depende qué hacer.

In [ ]:
faltantes = pd.DataFrame(
    {
        "faltantes": datos_limpios.isna().sum(),
        "porcentaje": (datos_limpios.isna().mean() * 100).round(1),
    }
)
print(faltantes[faltantes["faltantes"] > 0].to_string())

### Los tres mecanismos de datos faltantes

La distinción es de Rubin (1976) y determina qué imputación es válida:

| Mecanismo | Significa | Ejemplo |
|---|---|---|
| **MCAR** (completamente al azar) | La ausencia no depende de nada | Un sensor falló aleatoriamente |
| **MAR** (al azar condicionado) | La ausencia depende de **otras variables observadas** | Falta la edad más en tercera clase, y la clase sí la conocemos |
| **MNAR** (no al azar) | La ausencia depende del **propio valor ausente** | Los pasajeros sin camarote asignado no tienen cubierta |

Comprobemos si la ausencia de `age` es aleatoria o depende de algo.

In [ ]:
# Trabajamos sobre una copia auxiliar: todavía no modificamos el dataset.
aux = datos_limpios.assign(falta_edad=datos_limpios["age"].isna())

print("Proporción de edad faltante por clase:")
print(aux.groupby("pclass")["falta_edad"].mean().round(3).to_string())
print("\nProporción de edad faltante por puerto:")
print(aux.groupby("embarked")["falta_edad"].mean().round(3).to_string())
print("\nSupervivencia según si falta la edad:")
print(aux.groupby("falta_edad")["survived"].mean().round(3).to_string())

La ausencia de `age` **no es aleatoria en absoluto**:

- Falta en el 27.7 % de la tercera clase frente al 13.9 % de la primera.
- Y de forma mucho más marcada por puerto: falta en el **63.6 %** de quienes embarcaron en
  Queenstown, frente al 14 % de Southampton.
- Quienes tienen la edad ausente sobrevivieron menos (29.4 % frente a 40.6 %).

Es un caso de **MAR**: la ausencia depende de variables que sí observamos (la clase y el
puerto).

Esto tiene dos consecuencias prácticas:

1. Imputar con la media global sesgaría los datos: asignaría a los pasajeros de tercera
   clase una edad típica de toda la población. Es mejor imputar **por grupo** (por ejemplo,
   la mediana de su clase).
2. **El hecho de que falte es informativo en sí mismo.** Añadir una columna indicadora
   `falta_edad` conserva esa señal en vez de tirarla.

### El caso de `deck`

In [ ]:
print(f"deck: {datos_limpios['deck'].isna().mean():.1%} de valores faltantes\n")
print("Supervivencia según si se conoce la cubierta:")
print(
    datos_limpios.assign(tiene_deck=datos_limpios["deck"].notna())
    .groupby("tiene_deck")["survived"]
    .agg(["mean", "count"])
    .round(3)
    .to_string()
)
print("\nDistribución de clase según si se conoce la cubierta:")
print(
    pd.crosstab(datos_limpios["deck"].notna(), datos_limpios["pclass"], normalize="index")
    .round(3)
    .to_string()
)

Con un **77 % de valores faltantes**, `deck` parece candidata a eliminarse. Pero mira lo que
revelan los cruces: quienes tienen cubierta registrada sobrevivieron mucho más, y casi todos
son de primera clase.

La ausencia es **MNAR**: no se registró la cubierta porque muchos pasajeros de clases bajas
no tenían camarote asignado. El valor ausente depende del propio valor.

Aquí la decisión sensata no es imputar la cubierta —inventarse el 77 % de una columna es
inaceptable— sino **convertir la ausencia en información**: una variable binaria
"tiene camarote registrado".

> Este es el tipo de decisión que un `dropna()` automático destruye sin que nadie se entere.

In [ ]:
print("Valores faltantes en 'embarked': solo", datos_limpios["embarked"].isna().sum())
print(datos_limpios[datos_limpios["embarked"].isna()][["pclass", "sex", "age", "fare"]].to_string())

Dos filas. Con tan pocos casos, imputar con la moda (el puerto más frecuente) es razonable y
de bajo riesgo.

### Resumen de decisiones sobre faltantes

| Columna | % faltante | Mecanismo | Decisión |
|---|---|---|---|
| `age` | 19.9 % | MAR | Imputar por mediana **de la clase** + indicador `falta_edad` |
| `deck` | 77.2 % | MNAR | No imputar; transformar en binaria `tiene_camarote` |
| `embarked` | 0.2 % | MCAR (2 casos) | Imputar con la moda |

> **Cuidado.** Estas imputaciones **todavía no las aplicamos**. Deben calcularse usando solo
> el conjunto de entrenamiento, dentro de un `Pipeline`. Hacerlo ahora, sobre todo el
> dataset, sería fuga de datos — precisamente el tema del notebook 03.

## 4. Duplicados: cuidado con el automatismo

In [ ]:
print(f"Filas duplicadas (todas las columnas iguales): {datos_limpios.duplicated().sum()}")
duplicadas = datos_limpios[datos_limpios.duplicated(keep=False)].sort_values(
    ["pclass", "sex", "age", "fare"]
)
print(f"\nEjemplo de un grupo de filas idénticas:")
print(duplicadas.head(6)[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare"]].to_string())

Aparecen **107 filas duplicadas**. La reacción refleja sería `drop_duplicates()`. Sería un
error.

Este dataset **no tiene columna identificadora**: no hay nombre ni número de pasajero. Dos
hombres de tercera clase, ambos de 25 años, que viajaban solos y pagaron la misma tarifa
barata, producen filas idénticas **siendo personas distintas**.

La pregunta correcta no es "¿hay filas iguales?" sino **"¿hay registros repetidos?"**, y sin
un identificador no podemos responderla mirando la tabla.

| Situación | Acción |
|---|---|
| Hay ID y el ID se repite | Duplicado real: eliminar |
| Hay ID y el ID no se repite | Filas iguales por coincidencia: conservar |
| No hay ID | Investigar el origen de los datos antes de decidir |

**Decisión: conservarlas.** Eliminarlas descartaría 107 pasajeros reales y sesgaría el
dataset justo hacia el perfil más común (hombres jóvenes de tercera clase), que es además el
grupo con menor supervivencia.

## 5. Outliers: no todo valor extremo es un error

In [ ]:
def resumen_outliers(serie, nombre):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    fuera = ((serie < limite_inf) | (serie > limite_sup)).sum()
    print(
        f"{nombre:8s} Q1={q1:8.2f}  Q3={q3:8.2f}  IQR={iqr:8.2f}  "
        f"límites=[{limite_inf:.2f}, {limite_sup:.2f}]  outliers={fuera}"
    )


for col in ["age", "fare", "sibsp", "parch"]:
    resumen_outliers(datos_limpios[col].dropna(), col)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].boxplot([datos_limpios["age"].dropna()])
ejes[0].set_xticklabels(["age"])
ejes[0].set_title("Edad")
ejes[1].boxplot([datos_limpios["fare"]])
ejes[1].set_xticklabels(["fare"])
ejes[1].set_title("Tarifa")
plt.tight_layout()
plt.show()

Mirémoslos de cerca antes de juzgar.

In [ ]:
print("Las 5 edades más bajas:")
print(datos_limpios.nsmallest(5, "age")[["age", "who", "pclass", "survived"]].to_string(index=False))
print("\nLas 5 tarifas más altas:")
print(datos_limpios.nlargest(5, "fare")[["fare", "pclass", "who", "survived"]].to_string(index=False))
print(f"\nPasajeros con tarifa 0: {(datos_limpios['fare'] == 0).sum()}")
print(datos_limpios[datos_limpios["fare"] == 0]["pclass"].value_counts().to_string())

Tres tipos de valor extremo, con tres tratamientos distintos:

- **Edad 0.42** (un bebé de cinco meses): un valor perfectamente **legítimo**. La regla del
  IQR lo marca como outlier, y eliminarlo sería borrar a los bebés del dataset.
- **Tarifa 512.33** (tres pasajeros de primera clase): extrema pero **real**. Refleja la
  enorme desigualdad de precios a bordo, que es justamente parte de la señal que buscamos.
- **Tarifa 0** (15 pasajeros): esto sí es sospechoso. Probablemente sean tripulantes,
  invitados o un fallo de registro. Merece una nota, no un borrado automático.

> **La detección de outliers es estadística; la decisión sobre ellos es del dominio.** Un
> algoritmo puede señalarte los candidatos, pero solo alguien que entienda el problema puede
> decir si son errores o son la parte más interesante de los datos.

En este caso, `fare` tiene una asimetría fuerte. En vez de eliminar, lo razonable es
**transformar**: aplicar un logaritmo comprime la cola sin perder ninguna observación.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].hist(datos_limpios["fare"], bins=40)
ejes[0].set_title(f"fare (asimetría = {datos_limpios['fare'].skew():.2f})")
ejes[1].hist(np.log1p(datos_limpios["fare"]), bins=40)
ejes[1].set_title(f"log(1 + fare) (asimetría = {np.log1p(datos_limpios['fare']).skew():.2f})")
plt.tight_layout()
plt.show()

## 6. Análisis exploratorio

Ahora sí, la pregunta de fondo: ¿qué determinaba sobrevivir al Titanic?

In [ ]:
print(f"Tasa global de supervivencia: {datos_limpios['survived'].mean():.1%}\n")
for variable in ["sex", "pclass", "who", "alone", "embarked"]:
    tabla = datos_limpios.groupby(variable)["survived"].agg(["mean", "count"])
    tabla.columns = ["tasa_supervivencia", "n"]
    tabla["tasa_supervivencia"] = tabla["tasa_supervivencia"].round(3)
    print(f"--- {variable} ---")
    print(tabla.to_string())
    print()

El sexo es, con diferencia, el factor más determinante: sobrevivió el 74 % de las mujeres
frente al 19 % de los hombres. La clase también pesa mucho.

Pero lo interesante casi nunca está en las variables por separado, sino en su **interacción**.

In [ ]:
tabla_cruzada = datos_limpios.pivot_table(
    values="survived", index="sex", columns="pclass", aggfunc=["mean", "count"]
)
print("Tasa de supervivencia y número de pasajeros, por sexo y clase:")
print(tabla_cruzada.round(3).to_string())

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

# Supervivencia por sexo y clase.
pivote = datos_limpios.pivot_table(values="survived", index="pclass", columns="sex")
pivote.plot(kind="bar", ax=ejes[0], rot=0)
ejes[0].set_title("Supervivencia por clase y sexo")
ejes[0].set_ylabel("Tasa de supervivencia")
ejes[0].set_ylim(0, 1)

# Distribución de edad según supervivencia.
for valor, etiqueta in [(0, "No sobrevivió"), (1, "Sobrevivió")]:
    ejes[1].hist(
        datos_limpios.loc[datos_limpios["survived"] == valor, "age"].dropna(),
        bins=25, alpha=0.6, label=etiqueta,
    )
ejes[1].set_title("Edad y supervivencia")
ejes[1].set_xlabel("Edad")
ejes[1].legend()

# Tarifa (en log) según supervivencia.
ejes[2].boxplot(
    [
        np.log1p(datos_limpios.loc[datos_limpios["survived"] == 0, "fare"]),
        np.log1p(datos_limpios.loc[datos_limpios["survived"] == 1, "fare"]),
    ]
)
ejes[2].set_xticklabels(["No sobrevivió", "Sobrevivió"])
ejes[2].set_title("log(1 + tarifa) y supervivencia")

plt.tight_layout()
plt.show()

Tres lecturas:

- **La interacción es fuerte.** Una mujer de primera clase sobrevivió en el 97 % de los
  casos; un hombre de tercera, en el 14 %. El efecto del sexo no es el mismo en cada clase, y
  un modelo puramente aditivo no captará eso (lo retomaremos en la sesión 6).
- **Los niños pequeños sobrevivieron más**, coherente con la norma "mujeres y niños
  primero". La relación edad-supervivencia no es monótona: no basta con un coeficiente
  lineal.
- **Quien pagó más, sobrevivió más**, lo cual en buena medida es la clase por otra vía.

### Correlaciones entre variables numéricas

In [ ]:
numericas = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
correlacion = datos_limpios[numericas].corr().round(3)
print(correlacion.to_string())

`pclass` y `fare` están correlacionadas ($-0.55$): son dos formas de medir lo mismo. Cuando
dos predictoras están muy correlacionadas entre sí aparece la **multicolinealidad**, que
desestabiliza los coeficientes de un modelo lineal. Es el tema de la sesión 7.

También vale la pena notar que `sibsp` y `parch` miden ambas el tamaño del grupo familiar:
candidatas naturales a combinarse en una sola variable (sesión 5).

## 7. Conclusiones: la lista de decisiones

El EDA no termina en gráficas bonitas, termina en **decisiones justificadas**:

| Decisión | Justificación |
|---|---|
| Eliminar `alive` | Fuga de datos: es la variable objetivo |
| Eliminar `class`, `embark_town`, `adult_male` | Redundantes con otras columnas |
| Imputar `age` por mediana **de la clase** | Faltante MAR: depende de `pclass` |
| Añadir indicador `falta_edad` | La ausencia misma es informativa |
| Convertir `deck` en binaria `tiene_camarote` | 77 % faltante, mecanismo MNAR |
| Imputar `embarked` con la moda | Solo 2 casos |
| **No** eliminar duplicados | Sin ID, filas iguales ≠ registros repetidos |
| **No** eliminar outliers de `age` ni `fare` | Son valores legítimos |
| Transformar `fare` con logaritmo | Asimetría fuerte; comprime sin perder datos |
| Marcar `fare == 0` para revisión | 15 casos sospechosos |
| Combinar `sibsp` y `parch` | Miden lo mismo; se hará en el notebook 04 |

**Ninguna de estas transformaciones se aplica todavía.** Todas las que dependen de calcular
algo a partir de los datos (medianas, modas) deben calcularse **solo con el conjunto de
entrenamiento**. Por qué eso importa tanto es el tema del notebook 03; cómo se implementa
correctamente, el del notebook 04.

## Para practicar

1. ¿Depende la ausencia de `age` también del sexo o de viajar solo? Comprueba si hay más
   mecanismos MAR en juego.
2. Los 15 pasajeros con `fare == 0`: ¿de qué clase son y cuántos sobrevivieron? ¿Qué
   hipótesis sostendrías sobre quiénes eran?
3. Construye la tabla de supervivencia cruzando `who` con `pclass`. ¿Se cumple "mujeres y
   niños primero" en las tres clases por igual?
4. La variable `alone` se deriva de `sibsp` y `parch`. Verifícalo. ¿Sobrevivieron más los
   que viajaban solos o acompañados? ¿Cambia la respuesta al separar por sexo?